#  Identificación inicial del dataset


Fuente: Gran Encuesta Integrada de Hogares (GEIH) — DANE, tabla "Hogares" (microdatos de pobreza monetaria y pobreza extrema en Colombia).

Enlace del diccionario original: https://microdatos.dane.gov.co/index.php/catalog/908/data-dictionary/F7?file_name=Personas


In [ ]:
import pandas as pd

# Ruta del archivo fuente (ajustar si se sube a Google Drive en Colab)
RAW_PATH = "Hogares.csv"

df = pd.read_csv(RAW_PATH)
print(f"Filas: {df.shape[0]:,} | Columnas: {df.shape[1]}")
df.head()

## 2. Diccionario de datos (resumen)

| Columna | Significado | Tipo |
|---|---|---|
| directorio | Llave vivienda | numérico |
| secuencia_p | Llave hogar | numérico |
| mes | Mes de encuesta (1-12) | numérico |
| clase | 1 = Urbano, 2 = Rural | numérico |
| dominio | Ciudad capital / resto urbano / rural | categórico |
| dpto | Departamento | categórico |
| nper / npersug | Personas en el hogar / unidad de gasto | numérico |
| ingtotug / ingtotarr | Ingreso total unidad de gasto (sin/con imputación de arriendo) | numérico |
| ingpcug | Ingreso per cápita de la unidad de gasto | numérico |
| li | Línea de indigencia (pobreza extrema) | numérico |
| lp | Línea de pobreza | numérico |
| pobre | 0 = No pobre, 1 = Pobre | numérico (binario) |
| indigente | 0 = No indigente, 1 = Indigente | numérico (binario) |
| npobres / nindigentes | Número de personas pobres / indigentes en el hogar | numérico |
| fex_c | Factor de expansión anualizado (peso muestral) | numérico |
| fex_dpto | Factor de expansión departamental | numérico |
| p5090 | Tipo de tenencia de la vivienda | numérico (categórico) |
| p5100 | Cuota de amortización mensual | numérico |
| p5130 | Arriendo estimado mensual | numérico |
| ciudad_sinam | Ciudad sin área metropolitana | categórico |

Nota: el fex_c es clave porque este dataset es una muestra de hogares y cualquier conteo agregado debe ponderarse por este factor, no contar filas directamente.


In [ ]:
# Tipos de dato y memoria
df.info()


In [ ]:
# Valores nulos por columna
nulos = df.isnull().sum()
nulos = nulos[nulos > 0].sort_values(ascending=False)
nulos_pct = (nulos / len(df) * 100).round(2)
pd.DataFrame({"nulos": nulos, "% del total": nulos_pct})


**Lectura de los nulos encontrados:**
- `p5100` (cuota de amortización): nulo para quienes no están pagando la vivienda a crédito — esperado.
- `p5130` (arriendo estimado): nulo principalmente para quienes ya pagan arriendo real — esperado.
- `ciudad_sinam`: nulo para hogares que sí pertenecen a una ciudad capital (el campo solo aplica a "resto urbano") — esperado.

Ningún nulo aparece en las variables clave para el análisis de pobreza (`ingpcug`, `lp`, `li`, `pobre`, `indigente`, `fex_c`), por lo que la calidad del dataset para el propósito del proyecto es alta.


In [ ]:
# Duplicados por llave (vivienda + hogar)
duplicados = df.duplicated(subset=["directorio", "secuencia_p"]).sum()
print(f"Registros duplicados por (directorio, secuencia_p): {duplicados}")


In [ ]:
# Exploración rápida de categóricas relevantes
print("Departamentos únicos:", df["dpto"].nunique())
print(df["dpto"].value_counts().head(10))
print()
print("Dominios:", df["dominio"].unique())
print()
print("Clase (1=Urbano, 2=Rural):", df["clase"].value_counts().to_dict())


In [ ]:
# Estadísticas descriptivas de variables numéricas clave
df[["ingpcug", "lp", "li", "fex_c"]].describe()


## 3. Validación contra rangos oficiales del diccionario

Comparación rápida entre los rangos observados y los rangos documentados oficialmente por el DANE en el diccionario original, para confirmar que la extracción no introdujo corrupción de datos.


In [ ]:
rangos_oficiales = {
    "li": (182935.52974, 316815.18966),
    "lp": (308654.56309, 675919.2315),
    "fex_c": (1.204049701, 1472.4021064),
}

for col, (lo, hi) in rangos_oficiales.items():
    obs_lo, obs_hi = df[col].min(), df[col].max()
    dentro = (obs_lo >= lo * 0.99) and (obs_hi <= hi * 1.01)  # tolerancia mínima
    print(f"{col}: observado [{obs_lo:.2f}, {obs_hi:.2f}] | oficial [{lo:.2f}, {hi:.2f}] -> {'OK' if dentro else 'REVISAR'}")
